In [27]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

MODEL_PATH = r"C:\git\CleverCheck\server\my_model\hebert_model_download"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH, local_files_only=True)

model.eval()  # קריטי לבדיקות

def predict(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=128
    )

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits

    # אם זה regression (ציון רציף)
    score = logits.item()

    return score


# -------------------------
# בדיקות
# -------------------------
samples = [
    "התשובה נכונה ומדויקת מאוד",
    "התשובה חלקית ולא ברורה",
    "אין קשר לשאלה בכלל"
]

for s in samples:
    print(s)
    print("Score:", predict(s))
    print("-" * 40)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 8278.07it/s]

התשובה נכונה ומדויקת מאוד


Score: 0.8019910454750061
----------------------------------------
התשובה חלקית ולא ברורה
Score: 0.39667898416519165
----------------------------------------
אין קשר לשאלה בכלל
Score: 0.16000625491142273
----------------------------------------


In [ ]:


tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

model.eval()  # חובה

# -------------------------
# פונקציית בניית קלט
# -------------------------
def build_input(question, reference, student):
    return f"שאלה: {question} תשובת מורה: {reference} תשובת תלמיד: {student}"


# -------------------------
# חיזוי
# -------------------------
def predict(question, reference, student):
    text = build_input(question, reference, student)

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=128
    )

    with torch.no_grad():
        outputs = model(**inputs)

    # regression score
    return outputs.logits.item()


# -------------------------
# בדיקות לדוגמה
# -------------------------
tests = [
    {
        "question": "מה גורם ליום ולילה?",
        "reference": "סיבוב כדור הארץ סביב צירו.",
        "student": "סיבוב כדור הארץ סביב צירו.",
        "expected": "HIGH"
    },
    {
        "question": "מה גורם ליום ולילה?",
        "reference": "סיבוב כדור הארץ סביב צירו.",
        "student": "השמש זזה וגורמת ליום ולילה",
        "expected": "LOW"
    },
    {
        "question": "מה גורם ליום ולילה?",
        "reference": "סיבוב כדור הארץ סביב צירו.",
        "student": "כדור הארץ מסתובב והשמש לא זזה",
        "expected": "MEDIUM"
    }
]

print("START EVALUATION\n")

for t in tests:
    score = predict(t["question"], t["reference"], t["student"])

    print("Q:", t["question"])
    print("Student:", t["student"])
    print("Score:", round(score, 3))
    print("Expected:", t["expected"])
    print("-" * 50)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import numpy as np


tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

model.eval()


def build_input(q, ref, student):
    return f"שאלה: {q} תשובת מורה: {ref} תשובת תלמיד: {student}"


def predict(q, ref, student):
    text = build_input(q, ref, student)

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=128
    )

    with torch.no_grad():
        out = model(**inputs)

    return out.logits.item()


# -------------------------
# סט בדיקות לפי אורכים
# -------------------------
tests = [

    # 🟢 קצר ונכון
    {
        "type": "SHORT_CORRECT",
        "q": "מה גורם ליום ולילה?",
        "ref": "סיבוב כדור הארץ סביב צירו.",
        "student": "כדור הארץ מסתובב סביב עצמו."
    },

    # 🟢 קצר שגוי
    {
        "type": "SHORT_WRONG",
        "q": "מה גורם ליום ולילה?",
        "ref": "סיבוב כדור הארץ סביב צירו.",
        "student": "השמש מסתובבת סביב כדור הארץ."
    },

    # 🟡 בינוני נכון
    {
        "type": "MEDIUM_CORRECT",
        "q": "מה גורם ליום ולילה?",
        "ref": "סיבוב כדור הארץ סביב צירו.",
        "student": "יום ולילה נוצרים בגלל סיבוב כדור הארץ סביב צירו ביחס לשמש."
    },

    # 🟡 בינוני שגוי
    {
        "type": "MEDIUM_WRONG",
        "q": "מה גורם ליום ולילה?",
        "ref": "סיבוב כדור הארץ סביב צירו.",
        "student": "השמש זזה ומסתובבת סביב כדור הארץ וגורמת ליום ולילה."
    },

    # 🔵 ארוך נכון
    {
        "type": "LONG_CORRECT",
        "q": "מה גורם ליום ולילה?",
        "ref": "סיבוב כדור הארץ סביב צירו.",
        "student": "תופעת היום והלילה נגרמת כתוצאה מסיבוב כדור הארץ סביב צירו, כאשר חלקים שונים של הכוכב פונים לשמש וחווים אור, בעוד החלקים האחרים נמצאים בחושך."
    },

    # 🔵 ארוך שגוי
    {
        "type": "LONG_WRONG",
        "q": "מה גורם ליום ולילה?",
        "ref": "סיבוב כדור הארץ סביב צירו.",
        "student": "השמש היא זו שמסתובבת סביב כדור הארץ בתנועה יומית ולכן נוצר יום ולילה, כאשר היא נעה מצד אחד לשני של השמים."
    },

    # ⚫ רעש + ארוך
    {
        "type": "LONG_NOISY",
        "q": "מה גורם ליום ולילה?",
        "ref": "סיבוב כדור הארץ סביב צירו.",
        "student": "סיבוב כדור הארץ!!!! סביב הציר שלו??? גורם ליום ולילה בצורה מחזורית מאוד מעניינת!!! כאשר יש אור וחושך לסירוגין..."
    }
]


# -------------------------
# הרצה וסטטיסטיקה לפי סוג
# -------------------------
results = {}

print("START LENGTH EVALUATION\n")

for t in tests:
    score = predict(t["q"], t["ref"], t["student"])

    print(t["type"])
    print("SCORE:", round(score, 4))
    print("-" * 50)

    results.setdefault(t["type"], []).append(score)


print("\nSUMMARY\n")
for k, v in results.items():
    print(k, "AVG:", round(np.mean(v), 4))

In [ ]:
import pandas as pd
import re

file_path = r'C:\Users\levm\Desktop\train.csv'

df = pd.read_csv(file_path, encoding="utf-8-sig")

scores = []


import re

def parse(text):
    q_match = re.search(r"\[Q\](.*?)\[T\]", text)
    t_match = re.search(r"\[T\](.*?)\[S\]", text)
    s_match = re.search(r"\[S\](.*)", text)  # בלי פסיק!

    if not q_match or not t_match or not s_match:
        print("SKIP LINE (bad format):", text)
        return None

    q = q_match.group(1).strip()
    ref = t_match.group(1).strip()
    student = s_match.group(1).strip()

    return q, ref, student
for item in df["text"]:
    q, ref, student = parse(item)
    pred = predict(q, ref, student)
    scores.append(pred)

print("MIN:", min(scores))
print("MAX:", max(scores))
print("AVG:", sum(scores)/len(scores))

In [5]:
import pandas as pd
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import matplotlib.pyplot as plt
import re

model_path = r"C:\git\CleverCheck\server\my_model\my_trained_dictabert"

tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
model = AutoModelForSequenceClassification.from_pretrained(model_path, local_files_only=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()
print("num_labels =", model.config.num_labels)
print("problem_type =", model.config.problem_type)

def parse_row(text):
    q = re.search(r"\[Q\](.*?)\[T\]", text)
    t = re.search(r"\[T\](.*?)\[S\]", text)
    s = re.search(r"\[S\](.*)", text)

    return (
        q.group(1).strip() if q else "",
        t.group(1).strip() if t else "",
        s.group(1).strip() if s else ""
    )


def build_input(q, t, s):
    return f"שאלה: {q}\nתשובת מורה: {t}\nתשובת תלמיד: {s}"


def predict_batch(batch):
    texts = [build_input(q, t, s) for q, t, s in batch]

    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=256
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
       outputs = model(**inputs)

    return outputs.logits.cpu().numpy().flatten()


# ---------------------------
# LOAD CSV
# ---------------------------
file_path = 'C:\\git\\CleverCheck\\server\\services\\train.csv'


df = pd.read_csv(file_path, encoding="utf-8-sig")


y_true = df.iloc[:, 1].values  # הציון (עמודה שנייה)
texts = df.iloc[:, 0].values   # הטקסט המשולב

y_pred = []

batch_size = 32

for i in range(0, len(df), batch_size):
    chunk = texts[i:i+batch_size]

    batch = [parse_row(t) for t in chunk]

    preds = predict_batch(batch)
    y_pred.extend(preds)


y_true = np.array(y_true, dtype=float)
y_pred = np.array(y_pred)


# ---------------------------
# METRICS
# ---------------------------
mae = np.mean(np.abs(y_true - y_pred))
corr = np.corrcoef(y_true, y_pred)[0, 1]
acc = np.mean(np.abs(y_true - y_pred) < 0.2)

print("MAE:", mae)
print("Correlation:", corr)
print("Threshold accuracy:", acc)


# ---------------------------
# PLOTS
# ---------------------------
plt.figure()
plt.hist(y_pred, bins=30)
plt.title("Predicted Distribution")
plt.show()

plt.figure()
plt.scatter(y_true, y_pred, s=5)
plt.title("True vs Predicted")
plt.show()
print(y_pred.min())
print(y_pred.max())
print(y_pred.mean())

num_labels = 1
problem_type = regression


KeyboardInterrupt: 

In [5]:
print(y_true.min())
print(y_true.max())

print(np.percentile(y_true, 5))
print(np.percentile(y_true, 95))
print(y_pred.min())
print(y_pred.max())
print(y_pred.mean())
print(np.corrcoef(y_true, np.clip(y_pred, 0, 1))[0,1])

0.0
1.0
0.0
1.0
-3.658858
4.1970696
0.17675732
0.9026140747125069


In [3]:
import pandas as pd
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ---------------------------
# MODEL LOAD
# ---------------------------
model_path = r"C:\git\CleverCheck\server\my_model\hebert_model_download (2)"

tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
tokenizer.model_max_length = 512

model = AutoModelForSequenceClassification.from_pretrained(model_path, local_files_only=True)
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


# ---------------------------
# PREDICT
# ---------------------------
def predict(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    pred = torch.sigmoid(outputs.logits)

    return float(pred.cpu().numpy().flatten()[0])


# ---------------------------
# LOAD DATA
# ---------------------------
file_path = r"C:\git\CleverCheck\server\services\test.csv"
df = pd.read_csv(file_path, encoding="utf-8-sig")

texts = df.iloc[:, 0].astype(str).tolist()
y_true = df.iloc[:, 1].astype(float).tolist()

# ---------------------------
# PRINT RESULTS
# ---------------------------
print("\n===== FULL EVALUATION =====\n")

for i, (text, true_score) in enumerate(zip(texts, y_true)):
    pred = predict(text)

    print(f"#{i}")
    print("TRUE:", true_score)
    print("PRED:", round(pred, 4))
    print("TEXT:", text)
    print("-" * 50)


===== FULL EVALUATION =====

#0
TRUE: 1.0
PRED: 0.8399
TEXT: [Q] מה גורם ליום ולילה? [T] סיבוב כדור הארץ סביב עצמו [S] כדור הארץ מסתובב סביב עצמו
--------------------------------------------------
#1
TRUE: 0.2
PRED: 0.011
TEXT: [Q] מה גורם ליום ולילה? [T] סיבוב כדור הארץ סביב עצמו [S] השמש זזה
--------------------------------------------------
#2
TRUE: 0.6
PRED: 0.3449
TEXT: [Q] מהי מערכת השמש? [T] השמש וכוכבי הלכת שמקיפים אותה [S] השמש וכוכבים
--------------------------------------------------
#3
TRUE: 1.0
PRED: 0.4272
TEXT: [Q] מהי מערכת השמש? [T] השמש וכוכבי הלכת שמקיפים אותה [S] השמש וכוכבי הלכת
--------------------------------------------------
#4
TRUE: 0.8
PRED: 0.6041
TEXT: [Q] מהי פוטוסינתזה? [T] תהליך שבו צמחים משתמשים באור שמש ליצירת אנרגיה [S] צמחים משתמשים באור כדי לגדול
--------------------------------------------------
#5
TRUE: 0.4
PRED: 0.2022
TEXT: [Q] מהי פוטוסינתזה? [T] תהליך שבו צמחים משתמשים באור שמש ליצירת אנרגיה [S] תהליך בצמחים
----------------------------------

In [16]:
baseline = np.mean(y_true)
print(baseline)

0.5218174273858921


In [3]:
import pandas as pd
import re
import random

INPUT_FILE = r"C:\git\CleverCheck\server\services\test.csv"
OUTPUT_FILE = r"C:\git\CleverCheck\server\services\augmented_2000.csv"

df = pd.read_csv(INPUT_FILE, header=None, encoding="utf-8-sig")

augmented = []
seen_q = set()

# -------------------------
# parser יציב
# -------------------------
def safe_parse(line):
    try:
        if "," not in line:
            return None

        text, score = line.rsplit(",", 1)

        q_start = text.find("[Q]")
        t_start = text.find("[T]")
        s_start = text.find("[S]")

        if q_start == -1 or t_start == -1 or s_start == -1:
            return None

        q = text[q_start+3:t_start].strip()
        ref = text[t_start+3:s_start].strip()
        student = text[s_start+3:].strip()

        return q, ref, student

    except:
        return None


# -------------------------
# גיוון אמיתי (חשוב!)
# -------------------------
def weak_answer(q):
    return random.choice([
        "חלק מהמידע לא מדויק",
        "תשובה כללית מדי",
        "חסר פירוט בתשובה",
        "יש רק הבנה חלקית",
        "התשובה לא מלאה"
    ])

def shorten(text):
    words = text.split()
    return " ".join(words[:max(2, len(words)//3)])

def drop_info(text):
    words = text.split()
    if len(words) <= 3:
        return "תשובה חלקית"
    return " ".join(words[:-4])


# -------------------------
# יצירה
# -------------------------
TARGET = 2000

for line in df[0]:

    if len(augmented) >= TARGET:
        break

    parsed = safe_parse(line)
    if not parsed:
        continue

    q, ref, student = parsed

    # 🚨 מניעת חזרות
    if q in seen_q:
        continue
    seen_q.add(q)

    # 🟥 1. קיצור תשובה
    augmented.append([
        f"[Q] {q} [T] {ref} [S] {shorten(student)}",
        0.0
    ])

    if len(augmented) >= TARGET:
        break

    # 🟧 2. הורדת מידע
    augmented.append([
        f"[Q] {q} [T] {ref} [S] {drop_info(student)}",
        0.0
    ])

    if len(augmented) >= TARGET:
        break

    # 🟨 3. תשובה כללית אמיתית (לא חזרה)
    augmented.append([
        f"[Q] {q} [T] {ref} [S] {weak_answer(q)}",
        0.0
    ])


# -------------------------
# חיתוך ושמירה
# -------------------------
augmented = augmented[:TARGET]

df_out = pd.DataFrame(augmented)
df_out.to_csv(OUTPUT_FILE, index=False, header=False, encoding="utf-8-sig")

print("DONE ✔ 2000 דוגמאות מגוונות נוצרו בלי חזרות")

DONE ✔ 2000 דוגמאות מגוונות נוצרו בלי חזרות


In [4]:
import pandas as pd
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ---------------------------
# LOAD MODEL
# ---------------------------
model_path = r"C:\git\CleverCheck\server\my_model\my_trained_dictabert"

tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
tokenizer.model_max_length = 512

model = AutoModelForSequenceClassification.from_pretrained(model_path, local_files_only=True)
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# ---------------------------
# PREDICT
# ---------------------------
def predict(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.sigmoid(outputs.logits)
    return float(probs.cpu().numpy().flatten()[0])


# ---------------------------
# LOAD DATA
# ---------------------------
file_path = r"C:\git\CleverCheck\server\services\test.csv"
df = pd.read_csv(file_path, encoding="utf-8-sig")

texts = df.iloc[:, 0].astype(str).values
y_true = df.iloc[:, 1].astype(float).values

y_pred = []

# ---------------------------
# RUN PREDICTIONS
# ---------------------------
print("\nRunning predictions...\n")

for i in range(len(df)):
    pred = predict(texts[i])
    y_pred.append(pred)

    if i < 10:  # הדפסה לדיבוג ראשוני
        print(f"[{i}] TRUE={y_true[i]} PRED={round(pred,4)}")

y_pred = np.array(y_pred)

# ---------------------------
# METRICS
# ---------------------------
mae = np.mean(np.abs(y_true - y_pred))
corr = np.corrcoef(y_true, y_pred)[0, 1]
baseline = np.mean(y_true)

print("\n===== GLOBAL METRICS =====")
print("MAE:", mae)
print("Correlation:", corr)
print("Baseline:", baseline)
print("Model mean:", y_pred.mean())

# ---------------------------
# BUCKET ANALYSIS (חשוב מאוד!)
# ---------------------------
print("\n===== BUCKET ANALYSIS =====")

bins = np.linspace(0, 1, 6)  # 0-0.2-0.4-0.6-0.8-1.0
indices = np.digitize(y_true, bins)

for b in range(1, len(bins)):
    mask = indices == b
    if np.sum(mask) == 0:
        continue

    print(f"\nRange {bins[b-1]:.1f}-{bins[b]:.1f}")
    print("Count:", np.sum(mask))
    print("True mean:", np.mean(y_true[mask]))
    print("Pred mean:", np.mean(y_pred[mask]))

# ---------------------------
# EXTREME CHECK
# ---------------------------
print("\n===== EXTREMES CHECK =====")

low_mask = y_true <= 0.2
high_mask = y_true >= 0.8

print("\nLOW (0-0.2):")
print("True mean:", np.mean(y_true[low_mask]))
print("Pred mean:", np.mean(y_pred[low_mask]))

print("\nHIGH (0.8-1.0):")
print("True mean:", np.mean(y_true[high_mask]))
print("Pred mean:", np.mean(y_pred[high_mask]))


Running predictions...

[0] TRUE=1.0 PRED=0.728
[1] TRUE=0.2 PRED=0.5103
[2] TRUE=0.6 PRED=0.5427
[3] TRUE=1.0 PRED=0.6194
[4] TRUE=0.8 PRED=0.5951
[5] TRUE=0.4 PRED=0.5362
[6] TRUE=1.0 PRED=0.695
[7] TRUE=0.7 PRED=0.6626
[8] TRUE=1.0 PRED=0.6362
[9] TRUE=0.5 PRED=0.5564

===== GLOBAL METRICS =====
MAE: 0.25094134120723516
Correlation: 0.6826596153923769
Baseline: 0.5218174273858921
Model mean: 0.6313807455088588

===== BUCKET ANALYSIS =====

Range 0.0-0.2
Count: 126
True mean: 0.0007936507936507937
Pred mean: 0.5598288287246038

Range 0.2-0.4
Count: 303
True mean: 0.23366336633663365
Pred mean: 0.5708647613281465

Range 0.4-0.6
Count: 409
True mean: 0.5376528117359413
Pred mean: 0.6409538793388964

Range 0.6-0.8
Count: 54
True mean: 0.7009259259259262
Pred mean: 0.6591491235627068

Range 0.8-1.0
Count: 105
True mean: 0.8775238095238095
Pred mean: 0.6839797661418007

===== EXTREMES CHECK =====

LOW (0-0.2):
True mean: 0.12324159021406728
Pred mean: 0.5608545347638086

HIGH (0.8-1.0):


In [2]:
import pandas as pd
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ---------------------------
# LOAD MODEL
# ---------------------------
model_path = r"C:\git\CleverCheck\server\my_model\hebert_model_download (2)"

tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
tokenizer.model_max_length = 512

model = AutoModelForSequenceClassification.from_pretrained(model_path, local_files_only=True)
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# ---------------------------
# PREDICT
# ---------------------------
def predict(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.sigmoid(outputs.logits)
    return float(probs.cpu().numpy().flatten()[0])


# ---------------------------
# LOAD DATA
# ---------------------------
file_path = r"C:\git\CleverCheck\server\services\test.csv"
df = pd.read_csv(file_path, encoding="utf-8-sig")

texts = df.iloc[:, 0].astype(str).values
y_true = df.iloc[:, 1].astype(float).values

y_pred = []

# ---------------------------
# RUN PREDICTIONS
# ---------------------------
print("\nRunning predictions...\n")

for i in range(len(df)):
    pred = predict(texts[i])
    y_pred.append(pred)

    if i < 10:  # הדפסה לדיבוג ראשוני
        print(f"[{i}] TRUE={y_true[i]} PRED={round(pred,4)}")

y_pred = np.array(y_pred)

# ---------------------------
# METRICS
# ---------------------------
mae = np.mean(np.abs(y_true - y_pred))
corr = np.corrcoef(y_true, y_pred)[0, 1]
baseline = np.mean(y_true)

print("\n===== GLOBAL METRICS =====")
print("MAE:", mae)
print("Correlation:", corr)
print("Baseline:", baseline)
print("Model mean:", y_pred.mean())

# ---------------------------
# BUCKET ANALYSIS (חשוב מאוד!)
# ---------------------------
print("\n===== BUCKET ANALYSIS =====")

bins = np.linspace(0, 1, 6)  # 0-0.2-0.4-0.6-0.8-1.0
indices = np.digitize(y_true, bins)

for b in range(1, len(bins)):
    mask = indices == b
    if np.sum(mask) == 0:
        continue

    print(f"\nRange {bins[b-1]:.1f}-{bins[b]:.1f}")
    print("Count:", np.sum(mask))
    print("True mean:", np.mean(y_true[mask]))
    print("Pred mean:", np.mean(y_pred[mask]))

# ---------------------------
# EXTREME CHECK
# ---------------------------
print("\n===== EXTREMES CHECK =====")

low_mask = y_true <= 0.2
high_mask = y_true >= 0.8

print("\nLOW (0-0.2):")
print("True mean:", np.mean(y_true[low_mask]))
print("Pred mean:", np.mean(y_pred[low_mask]))

print("\nHIGH (0.8-1.0):")
print("True mean:", np.mean(y_true[high_mask]))
print("Pred mean:", np.mean(y_pred[high_mask]))


Running predictions...

[0] TRUE=1.0 PRED=0.8399
[1] TRUE=0.2 PRED=0.011
[2] TRUE=0.6 PRED=0.3449
[3] TRUE=1.0 PRED=0.4272
[4] TRUE=0.8 PRED=0.6041
[5] TRUE=0.4 PRED=0.2022
[6] TRUE=1.0 PRED=0.7893
[7] TRUE=0.7 PRED=0.3074
[8] TRUE=1.0 PRED=0.2374
[9] TRUE=0.5 PRED=0.5004

===== GLOBAL METRICS =====
MAE: 0.24003786777618388
Correlation: 0.5436950704154605
Baseline: 0.5218174273858921
Model mean: 0.5134795364541199

===== BUCKET ANALYSIS =====

Range 0.0-0.2
Count: 126
True mean: 0.0007936507936507937
Pred mean: 0.3331501828654418

Range 0.2-0.4
Count: 303
True mean: 0.23366336633663365
Pred mean: 0.31214495014596005

Range 0.4-0.6
Count: 409
True mean: 0.5376528117359413
Pred mean: 0.5046760612201574

Range 0.6-0.8
Count: 54
True mean: 0.7009259259259262
Pred mean: 0.5286391277418092

Range 0.8-1.0
Count: 105
True mean: 0.8775238095238095
Pred mean: 0.6799011273753075

===== EXTREMES CHECK =====

LOW (0-0.2):
True mean: 0.12324159021406728
Pred mean: 0.29260265144778685

HIGH (0.8-1.0

In [26]:
import pandas as pd

df = pd.read_csv(r"C:\git\CleverCheck\server\services\train.csv", encoding="utf-8-sig")

scores = df.iloc[:,1].astype(float)

# חלוקה ל־10 "רמות"
labels = (scores * 9).round().astype(int)

print("Counts per level:")
print(labels.value_counts().sort_index())

print("\nMean original score per level:")
for i in range(10):
    group = scores[labels == i]
    print(i, len(group), group.mean())

Counts per level:
score
0    2531
1     983
2    1357
3    1026
4    1504
5     453
6     612
7    1437
8    1645
9    3242
Name: count, dtype: int64

Mean original score per level:
0 2531 0.004243382062425919
1 983 0.10561546286876909
2 1357 0.20577008106116432
3 1026 0.30892787524366466
4 1504 0.45689494680851067
5 453 0.5888741721854304
6 612 0.6876960784313726
7 1437 0.7825748086290883
8 1645 0.8845045592705169
9 3242 0.9802899444787169


In [2]:
import pyodbc

conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=CleverCheckDB;"
    "Trusted_Connection=yes;"
)
print("connected")

connected


In [ ]:
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, BertModel

# -----------------------
# 1. טעינת מודל
# -----------------------
model_path = r"C:\git\CleverCheck\server\my_model\my_trained_dictabert"

tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
tokenizer.model_max_length = 512

model = AutoModelForSequenceClassification.from_pretrained(model_path, local_files_only=True)
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# -----------------------
# 2. parsing יציב
# -----------------------
def extract_parts(row):
    text, score = row.rsplit(",", 1)
    score = float(score)

    q = text.split("[Q]")[1].split("[T]")[0].strip()
    t = text.split("[T]")[1].split("[S]")[0].strip()
    s = text.split("[S]")[1].strip()

    return q, t, s, score


df2 = df["text"].astype(str).apply(extract_parts)
df2 = pd.DataFrame(df2.tolist(), columns=["Q", "T", "S", "score"])

# -----------------------
# 3. mean pooling
# -----------------------
def mean_pooling(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    return (last_hidden_state * mask).sum(1) / torch.clamp(mask.sum(1), min=1e-9)

# -----------------------
# 4. encode פונקציה
# -----------------------
def encode(texts):
    inputs = tokenizer(
        texts,
        padding=True,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    emb = mean_pooling(outputs.last_hidden_state, inputs["attention_mask"])
    emb = torch.nn.functional.normalize(emb, p=2, dim=1)
    return emb.cpu().numpy()

# -----------------------
# 5. embeddings
# -----------------------
t_emb = encode(df2["T"].tolist())
s_emb = encode(df2["S"].tolist())

# cosine similarity
pred_scores = np.sum(t_emb * s_emb, axis=1)

# -----------------------
# 6. reliability diagram
# -----------------------
def reliability_diagram(y_true, y_pred, bins=10):
    y_true = np.array(y_true).astype(float)
    y_pred = np.array(y_pred)

    bin_edges = np.linspace(0, 1, bins + 1)

    conf_list = []
    acc_list = []

    for i in range(bins):
        mask = (y_pred >= bin_edges[i]) & (y_pred < bin_edges[i + 1])
        if np.sum(mask) == 0:
            continue

        conf_list.append(np.mean(y_pred[mask]))
        acc_list.append(np.mean(y_true[mask]))

    plt.plot(conf_list, acc_list, marker="o")
    plt.plot([0, 1], [0, 1], "--")
    plt.xlabel("Confidence")
    plt.ylabel("Accuracy")
    plt.title("Reliability Diagram")
    plt.grid(True)
    plt.show()

# label בינארי מה-score המקורי
y_true = (df2["score"] >= 0.5).astype(int).values

reliability_diagram(y_true, pred_scores)

# -----------------------
# 7. בדיקות מהירות
# -----------------------
print("min:", pred_scores.min())
print("max:", pred_scores.max())

0.048920102417469025


In [2]:
from werkzeug.security import generate_password_hash

password = "4321"

hashed = generate_password_hash(password)

print(hashed)

scrypt:32768:8:1$8qHbWSG7Jftz9dUt$9fd8f0cfdfd1b8804724cc419a25a563af6c0d37731987a0a826fcfa385a50e4a551c84698c2c9b6e9606d834c5af665ee80078041e4055d3a1eff947cd8dc22
